In [1]:
import os
import numpy as np
import math
import time
import warnings
from itertools import permutations
from collections import defaultdict

# ── Qiskit core & simulator imports ──
from qiskit.circuit.library import QAOAAnsatz, GlobalPhaseGate
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as AerEstimator, SamplerV2 as AerSampler
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════
#  FLAGS — change these to control where things run
# ══════════════════════════════════════════════════════
USE_REAL_HARDWARE = True    # True = sample final circuits on QPU
USE_AER_FOR_OPT   = True    # True = optimize on Aer (recommended!)
USE_GPU_AER       = True    # True = try GPU-accelerated Aer; falls back to CPU if unavailable
                             # Requires: pip install qiskit-aer-gpu-cu11>=0.17
                             # (works on both CUDA 11 & 12 — the CUDA 12 package lacks a Qiskit 2.0 build)

# ── GPU Aer Options (shared across estimator & sampler) ──
_aer_gpu_options = {}
if USE_GPU_AER:
    try:
        from qiskit_aer import AerSimulator
        from qiskit import QuantumCircuit as _QC
        # Run a trivial 1-qubit circuit to verify GPU actually works
        # (AerSimulator init succeeds even without GPU — the error comes at runtime)
        _test_sim = AerSimulator(method='statevector', device='GPU')
        _test_qc = _QC(1); _test_qc.h(0); _test_qc.save_statevector()
        _test_sim.run(_test_qc).result()
        del _test_sim, _test_qc
        _aer_gpu_options = {'backend_options': {'device': 'GPU', 'method': 'statevector'}}
        print("Aer: GPU backend verified")
    except Exception as e:
        _aer_gpu_options = {}
        print(f"Aer GPU unavailable ({e}), falling back to CPU")

# ── Device ID ──
DEVICE_ID = "aws:ionq:qpu:forte-1"

# ── Connect via qBraid Runtime (Auto-transpiles from Qiskit!) ──
if USE_REAL_HARDWARE or not USE_AER_FOR_OPT:
    from qbraid.runtime import QbraidProvider

    provider = QbraidProvider()
    device = provider.get_device(DEVICE_ID)

    print(f"Connected to: {device.id}")
    print(f"Status:       {device.status()}")
    try:
        print(f"Queue depth:  {device.queue_depth()}")
    except Exception:
        print("Queue depth:  (not available)")
else:
    device = None

# ── Utility Functions ──
def euclidean_distance(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def build_distance_matrix(nodes):
    n = len(nodes)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i,j] = euclidean_distance(nodes[i], nodes[j])
    return D

def expectation_from_counts(counts, hamiltonian):
    """Fallback to compute <H> manually from counts (needed if optimizing on QPU)."""
    total = sum(counts.values())
    if total == 0:
        return 0.0
    exp_val = 0.0
    for pauli, coeff in zip(hamiltonian.paulis, hamiltonian.coeffs):
        label = pauli.to_label()
        z_positions = [i for i, c in enumerate(label) if c == 'Z']
        if not z_positions:
            exp_val += coeff.real
            continue
        term_exp = 0.0
        for bitstring, count in counts.items():
            eigenvalue = 1
            for pos in z_positions:
                eigenvalue *= (-1) ** int(bitstring[pos])
            term_exp += eigenvalue * count / total
        exp_val += coeff.real * term_exp
    return float(np.real(exp_val))


Aer GPU unavailable (Simulation device "GPU" is not supported on this system), falling back to CPU


ModuleNotFoundError: No module named 'qbraid'

In [15]:
def nn_route_cost(cluster, nodes):
    if not cluster: return 0
    route = [0]
    remaining = list(cluster)
    while remaining:
        last = route[-1]
        nearest = min(remaining, key=lambda c: math.sqrt((nodes[last][0]-nodes[c][0])**2 + (nodes[last][1]-nodes[c][1])**2))
        route.append(nearest)
        remaining.remove(nearest)
    cost = sum(math.sqrt((nodes[route[i]][0]-nodes[route[i+1]][0])**2 + (nodes[route[i]][1]-nodes[route[i+1]][1])**2) for i in range(len(route)-1))
    cost += math.sqrt((nodes[route[-1]][0]-nodes[0][0])**2 + (nodes[route[-1]][1]-nodes[0][1])**2)
    return cost

def two_opt_cost(cluster, nodes):
    if not cluster or len(cluster) <= 1:
        c = cluster[0] if cluster else 0
        return 2 * math.sqrt((nodes[0][0]-nodes[c][0])**2 + (nodes[0][1]-nodes[c][1])**2) if cluster else 0
    def d(a, b): return math.sqrt((nodes[a][0]-nodes[b][0])**2 + (nodes[a][1]-nodes[b][1])**2)
    route = [0]
    remaining = list(cluster)
    while remaining:
        last = route[-1]
        nearest = min(remaining, key=lambda c: d(last, c))
        route.append(nearest)
        remaining.remove(nearest)
    route.append(0)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(route) - 2):
            for j in range(i + 1, len(route) - 1):
                old = d(route[i-1], route[i]) + d(route[j], route[j+1])
                new = d(route[i-1], route[j]) + d(route[i], route[j+1])
                if new < old - 1e-10:
                    route[i:j+1] = reversed(route[i:j+1])
                    improved = True
    return sum(d(route[k], route[k+1]) for k in range(len(route)-1))

def agglomerative_clusters(nodes, Nv, C):
    n = len(nodes)
    D = build_distance_matrix(nodes)
    clusters = [[i] for i in range(1, n)]
    def nn_cost(cluster):
        if len(cluster) == 1: return 2 * D[0, cluster[0]]
        cost, current, remaining = 0, 0, set(cluster)
        while remaining:
            nxt = min(remaining, key=lambda c: D[current, c])
            cost += D[current, nxt]
            current = nxt
            remaining.remove(nxt)
        return cost + D[current, 0]
    while len(clusters) > 1:
        best_benefit, best_pair = -float('inf'), None
        for a in range(len(clusters)):
            for b in range(a + 1, len(clusters)):
                if len(clusters[a]) + len(clusters[b]) > C: continue
                benefit = nn_cost(clusters[a]) + nn_cost(clusters[b]) - nn_cost(clusters[a] + clusters[b])
                if benefit > best_benefit:
                    best_benefit = benefit
                    best_pair = (a, b)
        if best_pair is None: break
        if best_benefit <= 0 and len(clusters) <= Nv: break
        a, b = best_pair
        clusters[a] = clusters[a] + clusters[b]
        clusters.pop(b)
    return clusters

def refine_clusters_swap(clusters, nodes, C, func):
    clusters = [list(c) for c in clusters]
    improved = True
    iteration = 0
    while improved:
        improved = False
        iteration += 1
        for i in range(len(clusters)):
            for j in range(i+1, len(clusters)):
                old_cost = func(clusters[i], nodes) + func(clusters[j], nodes)
                best_gain, best_swap = 0, None
                for ai, a in enumerate(clusters[i]):
                    for bi, b in enumerate(clusters[j]):
                        ci_new = clusters[i][:ai] + [b] + clusters[i][ai+1:]
                        cj_new = clusters[j][:bi] + [a] + clusters[j][bi+1:]
                        if len(ci_new) > C or len(cj_new) > C: continue
                        new_cost = func(ci_new, nodes) + func(cj_new, nodes)
                        gain = old_cost - new_cost
                        if gain > best_gain:
                            best_gain, best_swap = gain, (ai, bi, ci_new, cj_new)
                for ai, a in enumerate(clusters[i]):
                    if len(clusters[j]) < C:
                        ci_new = clusters[i][:ai] + clusters[i][ai+1:]
                        cj_new = clusters[j] + [a]
                        if ci_new:
                            new_cost = func(ci_new, nodes) + func(cj_new, nodes)
                            gain = old_cost - new_cost
                            if gain > best_gain:
                                best_gain, best_swap = gain, (-1, -1, ci_new, cj_new)
                for bi, b in enumerate(clusters[j]):
                    if len(clusters[i]) < C:
                        cj_new = clusters[j][:bi] + clusters[j][bi+1:]
                        ci_new = clusters[i] + [b]
                        if cj_new:
                            new_cost = func(ci_new, nodes) + func(cj_new, nodes)
                            gain = old_cost - new_cost
                            if gain > best_gain:
                                best_gain, best_swap = gain, (-1, -1, ci_new, cj_new)
                if best_swap and best_gain > 1e-10:
                    clusters[i], clusters[j] = best_swap[2], best_swap[3]
                    improved = True
    return clusters


In [ ]:
def build_tsp_qubo(cluster_indices, dist_matrix):
    n = len(cluster_indices)
    N = n * n
    Q = np.zeros((N, N))
    idx = lambda i, s: i * n + s
    max_d = max(dist_matrix[a, b] for a in [0] + cluster_indices for b in [0] + cluster_indices if a != b)
    A = max_d * n * 1.5

    for i in range(n):
        for s in range(n): Q[idx(i,s), idx(i,s)] -= A
        for s in range(n):
            for t in range(s+1, n): Q[idx(i,s), idx(i,t)] += 2 * A

    for s in range(n):
        for i in range(n): Q[idx(i,s), idx(i,s)] -= A
        for i in range(n):
            for j in range(i+1, n): Q[idx(i,s), idx(j,s)] += 2 * A

    for i, ci in enumerate(cluster_indices):
        Q[idx(i,0), idx(i,0)]     += dist_matrix[0, ci]
        Q[idx(i,n-1), idx(i,n-1)] += dist_matrix[ci, 0]
        for j, cj in enumerate(cluster_indices):
            if i != j:
                for s in range(n - 1):
                    a, b = idx(i, s), idx(j, s+1)
                    if a <= b: Q[a, b] += dist_matrix[ci, cj]
                    else: Q[b, a] += dist_matrix[ci, cj]
    return Q

def qubo_to_ising(Q):
    n = Q.shape[0]
    constant, h, J = 0.0, np.zeros(n), {}
    for k in range(n):
        constant += Q[k, k] / 2.0
        h[k]     -= Q[k, k] / 2.0
    for k in range(n):
        for l in range(k+1, n):
            if abs(Q[k, l]) > 1e-12:
                constant += Q[k, l] / 4.0
                h[k] -= Q[k, l] / 4.0
                h[l] -= Q[k, l] / 4.0
                J[(k, l)] = Q[k, l] / 4.0

    pauli_list = []
    for k in range(n):
        if abs(h[k]) > 1e-12:
            lbl = ['I'] * n
            lbl[n - 1 - k] = 'Z'
            pauli_list.append((''.join(lbl), h[k]))
    for (k, l), coef in J.items():
        if abs(coef) > 1e-12:
            lbl = ['I'] * n
            lbl[n - 1 - k] = 'Z'
            lbl[n - 1 - l] = 'Z'
            pauli_list.append((''.join(lbl), coef))
    if not pauli_list:
        pauli_list = [('I' * n, 0.0)]
    return SparsePauliOp.from_list(pauli_list).simplify(), constant

def decode_bitstring(bitstring, n, cluster_indices, dist_matrix):
    bits = [int(b) for b in reversed(bitstring)]
    while len(bits) < n * n: bits.append(0)

    x = np.zeros((n, n), dtype=int)
    for i in range(n):
        for s in range(n): x[i, s] = bits[i * n + s]

    if not all(x[i, :].sum() == 1 for i in range(n)): return None, float('inf')
    if not all(x[:, s].sum() == 1 for s in range(n)): return None, float('inf')

    order = [0] * n
    for i in range(n): order[int(np.argmax(x[i, :]))] = cluster_indices[i]

    d = dist_matrix[0, order[0]]
    for k in range(n - 1): d += dist_matrix[order[k], order[k+1]]
    d += dist_matrix[order[-1], 0]
    return order, d


def build_tsp_qubo(cluster_indices, dist_matrix):
    n = len(cluster_indices)
    N = n * n
    Q = np.zeros((N, N))
    idx = lambda i, s: i * n + s
    max_d = max(dist_matrix[a, b] for a in [0] + cluster_indices for b in [0] + cluster_indices if a != b)
    A = max_d * n * 1.5

    for i in range(n):
        for s in range(n): Q[idx(i,s), idx(i,s)] -= A
        for s in range(n):
            for t in range(s+1, n): Q[idx(i,s), idx(i,t)] += 2 * A

    for s in range(n):
        for i in range(n): Q[idx(i,s), idx(i,s)] -= A
        for i in range(n):
            for j in range(i+1, n): Q[idx(i,s), idx(j,s)] += 2 * A

    for i, ci in enumerate(cluster_indices):
        Q[idx(i,0), idx(i,0)]     += dist_matrix[0, ci]
        Q[idx(i,n-1), idx(i,n-1)] += dist_matrix[ci, 0]
        for j, cj in enumerate(cluster_indices):
            if i != j:
                for s in range(n - 1):
                    a, b = idx(i, s), idx(j, s+1)
                    if a <= b: Q[a, b] += dist_matrix[ci, cj]
                    else: Q[b, a] += dist_matrix[ci, cj]
    return Q

def qubo_to_ising(Q):
    n = Q.shape[0]
    constant, h, J = 0.0, np.zeros(n), {}
    for k in range(n):
        constant += Q[k, k] / 2.0
        h[k]     -= Q[k, k] / 2.0
    for k in range(n):
        for l in range(k+1, n):
            if abs(Q[k, l]) > 1e-12:
                constant += Q[k, l] / 4.0
                h[k] -= Q[k, l] / 4.0
                h[l] -= Q[k, l] / 4.0
                J[(k, l)] = Q[k, l] / 4.0

    pauli_list = []
    for k in range(n):
        if abs(h[k]) > 1e-12:
            lbl = ['I'] * n
            lbl[n - 1 - k] = 'Z'
            pauli_list.append((''.join(lbl), h[k]))
    for (k, l), coef in J.items():
        if abs(coef) > 1e-12:
            lbl = ['I'] * n
            lbl[n - 1 - k] = 'Z'
            lbl[n - 1 - l] = 'Z'
            pauli_list.append((''.join(lbl), coef))
    if not pauli_list:
        pauli_list = [('I' * n, 0.0)]
    return SparsePauliOp.from_list(pauli_list).simplify(), constant

def decode_bitstring(bitstring, n, cluster_indices, dist_matrix):
    bits = [int(b) for b in reversed(bitstring)]
    while len(bits) < n * n: bits.append(0)

    x = np.zeros((n, n), dtype=int)
    for i in range(n):
        for s in range(n): x[i, s] = bits[i * n + s]

    if not all(x[i, :].sum() == 1 for i in range(n)): return None, float('inf')
    if not all(x[:, s].sum() == 1 for s in range(n)): return None, float('inf')

    order = [0] * n
    for i in range(n): order[int(np.argmax(x[i, :]))] = cluster_indices[i]

    d = dist_matrix[0, order[0]]
    for k in range(n - 1): d += dist_matrix[order[k], order[k+1]]
    d += dist_matrix[order[-1], 0]
    return order, d


def solve_tsp_qaoa(cluster_indices, dist_matrix, reps=3, restarts=2, warm_params=None):
    n = len(cluster_indices)
    if n == 1: return None, None, 0, 0, 0.0, [], 0.0

    t0 = time.time()

    Q = build_tsp_qubo(cluster_indices, dist_matrix)
    H, offset = qubo_to_ising(Q)
    ansatz_raw = QAOAAnsatz(H, reps=reps).decompose(reps=3)

    gammas = [(k+1)/reps * 0.75 for k in range(reps)]
    betas  = [(1-(k+1)/reps) * 0.75 for k in range(reps)]
    linear_guess = np.array(betas + gammas)

    best_params, best_energy = None, float('inf')
    convergence_history = []  # energy per restart: [(restart_idx, [e0, e1, ...]), ...]

    if USE_AER_FOR_OPT:
        # ── Fast local Aer simulation ──
        estimator = AerEstimator(options=_aer_gpu_options) if _aer_gpu_options else AerEstimator()
        _eval_log = []

        def cost_func(params):
            val = estimator.run([(ansatz_raw, H, params)]).result()[0].data.evs
            _eval_log.append(float(val))
            return val

        aer_label = "Aer GPU" if _aer_gpu_options else "Aer CPU"
        print(f"    Optimizing on {aer_label} ({restarts} restarts)...")
    else:
        # ── QPU / Cloud Simulator Fallback ──
        meas_ansatz = ansatz_raw.measure_all(inplace=False)
        hw_shots = 1000

        def cost_func(params):
            bound = meas_ansatz.assign_parameters(params)
            
            # --- SAME FIXES AS THE SAMPLING PHASE ---
            # 1. Strip explicit GlobalPhaseGate instructions and reset phase
            from qiskit.circuit.library import GlobalPhaseGate
            bound.data = [inst for inst in bound.data if not isinstance(inst.operation, GlobalPhaseGate)]
            bound.global_phase = 0
            
            # 2. Submit to device
            job = device.run(bound, shots=hw_shots)
            
            # 3. Wait for completion safely
            job.wait_for_final_state()
            
            if job.status().name != 'COMPLETED':
                print(f"    Optimization Iteration Failed! Status: {job.status().name}")
                return 0.0  # Return poor energy rather than crashing
                
            result = job.result()
            
            # 4. Robust counts extraction
            counts = {}
            if hasattr(result, 'data') and hasattr(result.data, 'get_counts'):
                counts = result.data.get_counts()
            elif hasattr(result, 'get_counts'):
                counts = result.get_counts()
            elif hasattr(result, 'measurement_counts'):
                counts = getattr(result, "measurement_counts")
                if callable(counts): counts = counts()
                
            return expectation_from_counts(counts, H)

        print(f"    Optimizing on {device.id} ({restarts} restarts)...")
        print(f"    ⚠ Warning: Each iteration joins the queue. This takes hours.")

    for r in range(restarts):
        if r == 0:
            x0 = linear_guess; init_type = "Linear Ramp"
        elif r == 1 and warm_params is not None and len(warm_params) == ansatz_raw.num_parameters:
            x0 = warm_params.copy(); init_type = "Warm Start "
        else:
            x0 = np.random.uniform(-np.pi, np.pi, ansatz_raw.num_parameters); init_type = "Random     "

        if USE_AER_FOR_OPT:
            _eval_log.clear()
        res = minimize(cost_func, x0, method='COBYLA', options={'maxiter': 500})
        if USE_AER_FOR_OPT:
            convergence_history.append((init_type.strip(), list(_eval_log)))
        print(f"      Restart {r} [{init_type}]: Energy = {res.fun:8.2f} | Iters = {res.nfev}")
        if res.fun < best_energy:
            best_energy = res.fun
            best_params = res.x

    elapsed = time.time() - t0
    decomposed = ansatz_raw.decompose()
    return ansatz_raw, best_params, ansatz_raw.num_qubits, sum(decomposed.count_ops().values()), elapsed, convergence_history


In [17]:
# ── Configuration ──
selected_id = "6"  # Instance ID from final_instances.json

with open('final_instances.json', 'r') as f:
    all_instances = json.load(f)

config = next((inst for inst in all_instances if inst['instance_id'] == str(selected_id)), None)
assert config is not None, f"Instance '{selected_id}' not found in final_instances.json"

Nv = config['Nv']
C  = config['C']
nodes = [(c['x'], c['y']) for c in config['customers']]

print(f"Loaded Instance {selected_id}: {len(nodes)-1} customers, {Nv} vehicles, Capacity {C}, Max qubits: {C**2}")

Loaded Instance 4: 12 customers, 4 vehicles, Capacity 3, Max qubits: 9


In [ ]:
# 1. Distance matrix
D = build_distance_matrix(nodes)

# 2. Classical Clustering
clusters = agglomerative_clusters(nodes, Nv, C)
clusters = refine_clusters_swap(clusters, nodes, C, two_opt_cost)
print(f"\nClusters: {clusters}\n")

# 3. QAOA Optimization Phase
cluster_results = []
first_params = None
total_time = 0

for i, cluster in enumerate(clusters):
    print(f"  Cluster {i+1}/{len(clusters)}: {cluster}")
    ansatz, best_params, nq, ng, et, conv_hist, ising_offset = solve_tsp_qaoa(
        cluster, D, reps=3, restarts=2, warm_params=first_params
    )
    if i == 0 and best_params is not None:
        first_params = best_params.copy()

    cluster_results.append({
        'cluster': cluster, 'ansatz': ansatz, 'params': best_params,
        'nq': nq, 'ng': ng, 'convergence': conv_hist, 'offset': ising_offset
    })
    total_time += et

# 4. Sampling Phase
hw_label = device.id if USE_REAL_HARDWARE else ("Aer GPU" if _aer_gpu_options else "Aer CPU")
print(f"\n--- Sampling on {hw_label} ---")

shots = 1000 # Lower shots for IonQ since it bills by the shot
all_counts = {}

for i, cr in enumerate(cluster_results):
    if cr['ansatz'] is None:
        continue

    meas = cr['ansatz'].measure_all(inplace=False)
    bound = meas.assign_parameters(cr['params'])

    if USE_REAL_HARDWARE:
        # Actively strip explicit GlobalPhaseGate instructions and reset phase
        bound.data = [inst for inst in bound.data if not isinstance(inst.operation, GlobalPhaseGate)]
        bound.global_phase = 0
        
        print(f"  Submitting circuit {i+1} to {device.id}...")
        job = device.run(bound, shots=shots)
        
        job.wait_for_final_state()
        
        if job.status().name != 'COMPLETED':
            print(f"  ❌ Job Failed! Status: {job.status().name}")
            if hasattr(job, 'metadata'): print(f"  Metadata: {job.metadata()}")
            counts = {}  
        else:
            result = job.result()
            
            # Robust counts extraction
            counts = {}
            if hasattr(result, 'data') and hasattr(result.data, 'get_counts'):
                counts = result.data.get_counts()
            elif hasattr(result, 'get_counts'):
                counts = result.get_counts()
            elif hasattr(result, 'measurement_counts'):
                counts = getattr(result, "measurement_counts")
                if callable(counts): counts = counts()


# 5. Decode results
total_distance = 0
routes = []
max_qubits = 0
total_gates = 0

for i, cr in enumerate(cluster_results):
    cluster = cr['cluster']
    n = len(cluster)

    if cr['ansatz'] is None:
        route = list(cluster)
        dist = D[0, cluster[0]] * 2
    else:
        counts = all_counts.get(i, {})

        best_route, best_dist = None, float('inf')
        for bs in counts:
            r, d = decode_bitstring(bs, n, cluster, D)
            if r is not None and d < best_dist:
                best_dist = d
                best_route = r

        if best_route is None:
            for perm in permutations(cluster):
                d = D[0, perm[0]]
                for k in range(len(perm) - 1): d += D[perm[k], perm[k+1]]
                d += D[perm[-1], 0]
                if d < best_dist:
                    best_dist = d
                    best_route = list(perm)
        route, dist = best_route, best_dist

    full_route = [0] + route + [0]
    routes.append(full_route)
    total_distance += dist
    max_qubits = max(max_qubits, cr['nq'])
    total_gates += cr['ng']
    print(f"  r{i+1}: {' → '.join(map(str, full_route))}  |  Distance: {dist:.2f}  |  Qubits: {cr['nq']}")

print(f"\n{'='*50}")
print(f"Instance {selected_id}  —  Total Distance: {total_distance:.2f}")
print(f"Max Qubits: {max_qubits}  |  Total Gates: {total_gates}  |  Time: {total_time:.2f}s")
print(f"Backend: {hw_label}")
print(f"{'='*50}")


In [ ]:
# ── Cumulative Objective Convergence ──
import matplotlib.pyplot as plt
import csv

non_trivial = [cr for cr in cluster_results if cr['convergence']]
if not non_trivial:
    print("No convergence data (all clusters trivial or hardware-optimized)")
else:
    # Build a sequential trace: clusters optimized one after another
    # x = cumulative circuit evaluations, y = cumulative energy (running min per cluster)
    x_evals = []
    y_energy = []
    cumulative_evals = 0
    settled_energy = 0.0

    for cr in non_trivial:
        best_restart = min(cr['convergence'], key=lambda x: x[1][-1] if x[1] else float('inf'))
        trace = best_restart[1]

        curr_min = float('inf')
        for e in trace:
            cumulative_evals += 1
            curr_min = min(curr_min, e)
            x_evals.append(cumulative_evals)
            y_energy.append(settled_energy + curr_min)

        settled_energy += curr_min

    # Save to CSV
    conv_csv = f"convergence_{selected_id}.csv"
    with open(conv_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['circuit_evaluation', 'energy'])
        for x, y in zip(x_evals, y_energy):
            writer.writerow([x, round(y, 4)])
    print(f"Saved convergence data to {conv_csv}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x_evals, y_energy, color='tab:blue', linewidth=1.5)
    ax.set_title(f"QAOA Convergence — Instance {selected_id}")
    ax.set_xlabel("Total Circuit Evaluations")
    ax.set_ylabel("Cumulative Energy")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()